In [1]:
import os, sys, json, zipfile, hashlib, traceback
from pathlib import Path
run_id = 'V005_N1_N2_CPU_A003'
root = Path('/content/mla_task_runs/V005_N1_N2_CPU_A003')
stages = {'run_id':run_id,'setup':False,'data':False,'cuda':False,'training':False}
assert not (root/'execution_started.json').exists(), 'Attempt already executed; create a new run ID'
root.mkdir(parents=True,exist_ok=True)
bundle = Path('/content/mla_task_bundles/V005_N1_N2_CPU_A003.zip')
with zipfile.ZipFile(bundle) as archive: archive.extractall(root/'source')
sys.path.insert(0,str(root/'source'))
for name in list(sys.modules):
    if name == 'mla_exp' or name.startswith('mla_exp.') or name in ['span_model','sequence_model','llm_pilot','llm_route','hybrid_model','final_evaluation','assessment']: del sys.modules[name]
config = json.loads((root/'source/config.json').read_text())
import span_model
span_model.SETTINGS.update(config)
from mla_exp.common import write_json
if config.get('parent_checkpoint'):
    import shutil
    parent = Path(config['parent_checkpoint'])
    assert parent.is_absolute() and str(parent).startswith('/content/mla_task_runs/')
    assert (parent/'training.pt').exists() and (parent/'model/span_head.json').exists()
    parent_run = parent.parent.parent
    target = Path(config['output_root'])/'runs'/run_id
    target.mkdir(parents=True,exist_ok=False)
    child_checkpoint = target/'checkpoints'/parent.name
    shutil.copytree(parent,child_checkpoint)
    shutil.copytree(parent_run/'best',target/'best')
    shutil.copy2(parent_run/'metadata.json',target/'metadata.json')
    write_json(target/'resume.json',{'checkpoint':str(child_checkpoint),'parent':str(parent)})
write_json(root/'execution_started.json',{'run_id':run_id,'source_sha256':hashlib.sha256(bundle.read_bytes()).hexdigest(),'config':config})
print('CONFIG',json.dumps(config))
stages['setup'] = True


CONFIG {"seed": 42, "training_seeds": [42, 43, 44], "budget_cu": 50.0, "workload_cu": 45.0, "deadline_hours": 48, "data_root": "/content/mla/data", "output_root": "/content/mla_task_runs/V005_N1_N2_CPU_A003/artifacts", "drive_root": "/content/drive/MyDrive/MLA_MIDTERM", "max_features": 15000, "min_df": 2, "context_chars": 1800, "c_grid": [0.1, 1.0, 10.0], "class_weights": [null, "balanced"], "embedding_weights": [0.25, 1.0, 4.0], "cv_jobs": 1, "encoder": "microsoft/deberta-v3-base", "max_length": 512, "batch_size": 8, "effective_batch_size": 32, "epochs": 3, "checkpoint_steps": 500, "checkpoint_minutes": 15, "bootstrap_repeats": 2000, "minimum_slice_size": 100, "minimum_slice_class_size": 20, "llm_model": "Qwen/Qwen2.5-7B-Instruct", "llm_fractions": [0.02, 0.05, 0.1], "ensemble_weights": [0.25, 0.5, 0.75], "gates": {"macro_f1": 0.5, "recall_at_precision": 0.6, "precision": 0.4, "hybrid_gain": 0.02}, "stage_caps": {"prepare_hybrid": 5, "neural": 15, "confirmation": 12, "extension": 8, "

In [2]:
from mla_exp.data import load_frame, assert_disjoint
from mla_exp import LABELS
assert stages['run_id']==run_id and stages['setup'], 'Setup did not pass'
assert list(LABELS) == ['Ineffective','Adequate','Effective'], LABELS
development = load_frame(config['data_root'], 'development', config)
assert_disjoint(development)
for row in development.sample(min(100,len(development)),random_state=42).to_dict('records'):
    assert ' '.join(row['full_text'][int(row['discourse_start']):int(row['discourse_end'])].split()) == ' '.join(row['discourse_text'].split())
print('DATA_CONTRACT_PASS', len(development), development.groupby('split').group_id.nunique().to_dict())
stages['data'] = True


DATA_CONTRACT_PASS 144264 {'calibration': 1567, 'fit': 12468, 'validation': 1557}


In [3]:
assert stages['run_id']==run_id and stages['data'], 'Data contract did not pass'
import torch
torch.set_num_threads(1)
print('CPU_ENSEMBLE: aligned saved validation/calibration predictions only; no CUDA inference')
stages['cuda'] = True


CPU_ENSEMBLE: aligned saved validation/calibration predictions only; no CUDA inference


In [4]:
assert stages['run_id']==run_id and stages['cuda'], 'CUDA contract did not pass'
from final_evaluation import choose_ensemble
result=choose_ensemble(config['source_best'],config['source_b'],config,Path(config['output_root'])/'runs'/run_id)
print('ENSEMBLE_VALIDATION',json.dumps(result))
write_json(root/'status.json',{'status':'model_ready','run_id':run_id})

stages['training'] = True


ENSEMBLE_VALIDATION {"macro_f1": 0.6022433747094398, "accuracy": 0.7764804586450395, "qwk": 0.478412177696937, "precision_ineffective": 0.33517089305402425, "recall_ineffective": 0.47874015748031495, "per_class": {"Ineffective": {"precision": 0.33517089305402425, "recall": 0.47874015748031495, "f1": 0.3942931258106355, "support": 635}, "Adequate": {"precision": 0.8827300150829562, "recall": 0.8288926263609808, "f1": 0.8549646199497831, "support": 11297}, "Effective": {"precision": 0.5157819225251076, "recall": 0.606495149725854, "f1": 0.5574723783679008, "support": 2371}}, "n": 14303, "confusion_matrix": [[304, 320, 11], [594, 9364, 1339], [9, 924, 1438]], "log_loss": 0.5111913052296829, "brier_sum_classes": 0.312065667560362, "ece_10_bins": 0.009863611099907233, "recall_at_precision_0.4": 0.3889763779527559, "average_precision_ineffective": 0.3837721477829264, "average_precision_per_class": [0.3837721477829264, 0.9230324880557856, 0.600341960028034], "macro_average_precision": 0.63571

In [5]:
assert stages['run_id']==run_id and stages['training'], 'Training did not complete'
from mla_exp.evaluation import evaluate_file
for p in Path(config['output_root']).glob('runs/'+run_id+'/predictions_*.csv.gz'):
    report = p.parent/('evaluation_'+p.name.removeprefix('predictions_').removesuffix('.csv.gz'))/'report.json'
    evaluate_file(p,report,repeats=2000,data_root=config['data_root'])
    print('EVALUATION',report.name)
for p in Path(config['output_root']).glob('runs/*/metrics*.json'):
    print(p.name,p.read_text())

write_json(root/'status.json',{'status':'complete','run_id':run_id})
print('RUN_COMPLETE',run_id)


EVALUATION report.json


EVALUATION report.json
metrics_validation.json {
  "macro_f1": 0.6022433747094398,
  "accuracy": 0.7764804586450395,
  "qwk": 0.478412177696937,
  "precision_ineffective": 0.33517089305402425,
  "recall_ineffective": 0.47874015748031495,
  "per_class": {
    "Ineffective": {
      "precision": 0.33517089305402425,
      "recall": 0.47874015748031495,
      "f1": 0.3942931258106355,
      "support": 635
    },
    "Adequate": {
      "precision": 0.8827300150829562,
      "recall": 0.8288926263609808,
      "f1": 0.8549646199497831,
      "support": 11297
    },
    "Effective": {
      "precision": 0.5157819225251076,
      "recall": 0.606495149725854,
      "f1": 0.5574723783679008,
      "support": 2371
    }
  },
  "n": 14303,
  "confusion_matrix": [
    [
      304,
      320,
      11
    ],
    [
      594,
      9364,
      1339
    ],
    [
      9,
      924,
      1438
    ]
  ],
  "log_loss": 0.5111913052296829,
  "brier_sum_classes": 0.312065667560362,
  "ece_10_bins": 0.00